# Manual Bernoulli Naive Bayes for MNIST (7 vs Not-7)

This notebook implements a **binary classifier** that predicts whether a handwritten digit image is **7** or **not 7**, using a manually coded **Bernoulli Naive Bayes** model.

**Key points:**
- The MNIST dataset is loaded via `sklearn.datasets.fetch_openml`.
- The Naive Bayes model is implemented **from scratch** using NumPy.
- Features are binarized (pixel on/off) to match the Bernoulli distribution assumption.
- Laplace smoothing is applied to avoid zero-probability issues.
- Evaluation uses accuracy, precision, recall, F1 score, and a confusion matrix.

Before running the notebook, install the required packages if needed:

`pip install numpy scikit-learn notebook`

---
## 1. Imports

We import NumPy for numerical operations, `fetch_openml` to load MNIST, and several metrics from scikit-learn for evaluation.

In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

---
## 2. Load Data

We load the MNIST dataset (70,000 images of 28×28 handwritten digits) using `fetch_openml`.

In [2]:
print("Loading MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data.astype(float)
y = mnist.target.astype(int)

print(f"Dataset shape: X={X.shape}, y={y.shape}")

Loading MNIST...
Dataset shape: X=(70000, 784), y=(70000,)


---
## 3. Create Binary Labels (7 vs Not-7)

We convert the multiclass labels into a binary classification task:
- **1** → digit is 7
- **0** → digit is not 7

This produces an **imbalanced** dataset (~10% positive, ~90% negative). The model does **not** explicitly handle class imbalance; however, Bernoulli Naive Bayes naturally incorporates prior probabilities, and the 784-dimensional binary features provide sufficient discriminative power.

In [3]:
y = (y == 7).astype(int)

print("Class distribution:")
print("Digit 7:", np.sum(y == 1))
print("Not 7:", np.sum(y == 0))

Class distribution:
Digit 7: 7293
Not 7: 62707


---
## 4. Binarize Features

Bernoulli Naive Bayes assumes that each feature is binary (0 or 1). We:
1. Normalize pixel values to [0, 1] by dividing by 255.
2. Apply a threshold of **0.3** to convert each pixel into a binary on/off indicator.

The threshold can be tuned (typically between 0.2 and 0.5).

In [4]:
X = X / 255.0
X = (X > 0.3).astype(int)

print(f"Binarized features — unique values: {np.unique(X)}")
print(f"Feature matrix shape: {X.shape}")

Binarized features — unique values: [0 1]
Feature matrix shape: (70000, 784)


---
## 5. Train / Test Split

We split the data into **80% training** and **20% test** sets. The `stratify` parameter preserves the class ratio in both splits.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(f"  Digit 7: {np.sum(y_train == 1)}")
print(f"  Not 7:   {np.sum(y_train == 0)}")
print(f"\nTest class distribution:")
print(f"  Digit 7: {np.sum(y_test == 1)}")
print(f"  Not 7:   {np.sum(y_test == 0)}")

Training set: 56000 samples
Test set:     14000 samples

Training class distribution:
  Digit 7: 5834
  Not 7:   50166

Test class distribution:
  Digit 7: 1459
  Not 7:   12541


---
## 6. Manual Bernoulli Naive Bayes Implementation

### Mathematical Background

For each class $c$, we compute:

- **Prior probability:** $P(C = c) = \frac{N_c}{N}$

- **Feature likelihood (with Laplace smoothing):**
$$P(x_j = 1 \mid C = c) = \frac{\sum_{i \in c} x_{ij} + 1}{N_c + 2}$$

- **Log-posterior for prediction:**
$$\log P(C = c \mid \mathbf{x}) \propto \log P(C = c) + \sum_{j=1}^{d} \left[ x_j \log p_{cj} + (1 - x_j) \log(1 - p_{cj}) \right]$$

The predicted class is the one with the highest log-posterior.

In [6]:
class BernoulliNaiveBayes:
    def __init__(self):
        self.classes = None
        self.feature_prob = {}
        self.prior = {}

    def fit(self, X, y):
        self.classes = np.unique(y)

        for c in self.classes:
            X_c = X[y == c]

            # Laplace smoothing: add 1 to numerator, 2 to denominator
            self.feature_prob[c] = (np.sum(X_c, axis=0) + 1) / (len(X_c) + 2)
            self.prior[c] = len(X_c) / len(X)

    def predict(self, X):
        predictions = []

        for x in X:
            posteriors = []

            for c in self.classes:
                p = self.feature_prob[c]

                log_likelihood = np.sum(
                    x * np.log(p) + (1 - x) * np.log(1 - p)
                )

                log_prior = np.log(self.prior[c])
                posteriors.append(log_prior + log_likelihood)

            predictions.append(self.classes[np.argmax(posteriors)])

        return np.array(predictions)

---
## 7. Train the Model

In [7]:
print("Training model...")
model = BernoulliNaiveBayes()
model.fit(X_train, y_train)
print("Training complete.")

Training model...
Training complete.


---
## 8. Predict on the Test Set

In [8]:
print("Predicting on the test set...")
y_pred = model.predict(X_test)
print("Prediction complete.")

Predicting on the test set...
Prediction complete.


---
## 9. Evaluation

We evaluate the model using standard binary classification metrics:

| Metric | Description |
|--------|-------------|
| **Accuracy** | Overall correct predictions / total predictions |
| **Precision** | True positives / (True positives + False positives) |
| **Recall** | True positives / (True positives + False negatives) |
| **F1 Score** | Harmonic mean of precision and recall |

In [9]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

---
## 10. Results

In [10]:
print("===== RESULTS =====")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("\nConfusion Matrix:")
print(conf_matrix)

===== RESULTS =====
Accuracy : 0.9319
Precision: 0.6146
Recall   : 0.928
F1 Score : 0.7395

Confusion Matrix:
[[11692   849]
 [  105  1354]]
